# AutoML Tutorial: H2O AutoML
> **Run this notebook in Google Colab or Jupyter to learn about automated machine learning (AutoML) using H2O AutoML.**

## 1. Introduction
Automated Machine Learning (AutoML) simplifies the end-to-end ML workflow by:
- **Feature preprocessing**
- **Model selection**
- **Hyperparameter optimization**
- **Ensembling**

In this tutorial, we'll focus solely on **H2O AutoML**, a scalable AutoML framework supporting both regression and classification.


At the end, you'll complete an exercise applying AutoML.



## 2. Setup & Installation

In [2]:
!pip install --quiet jedi
!pip install --quiet h2o
!pip install --quiet 'thinc<8.3.6'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.4/266.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 47.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spacy 3.8.14 requires confection<2.0.0,>=1.3.2, but you have confection 0.1.5 which is incompatible.
spacy 3.8.14 requires thinc<8.4.0,>=8.3.12, but you have thinc 8.3.4 which is incompatible.
weasel 1.0.0 requires confection>=1.0.0, but you have confection 0.1.5 which is incompatible.


## Regression on California Housing


In [3]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 3. Regression Example: California Housing

In [4]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import h2o
from h2o.automl import H2OAutoML
from sklearn.metrics import mean_squared_error, r2_score

In [5]:
# Initialize H2O
h2o.init(max_mem_size="2G", nthreads=-1)

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "17.0.19" 2026-04-21; OpenJDK Runtime Environment (build 17.0.19+10-1-22.04.2-Ubuntu); OpenJDK 64-Bit Server VM (build 17.0.19+10-1-22.04.2-Ubuntu, mixed mode, sharing)
  Starting server from /usr/local/lib/python3.12/dist-packages/h2o/backend/bin/h2o.jar
  Ice root: /tmp/tmpd1jo9yja
  JVM stdout: /tmp/tmpd1jo9yja/h2o_unknownUser_started_from_python.out
  JVM stderr: /tmp/tmpd1jo9yja/h2o_unknownUser_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,03 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.11
H2O_cluster_version_age:,2 months and 6 days
H2O_cluster_name:,H2O_from_python_unknownUser_y7p6pn
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,1.980 Gb
H2O_cluster_total_cores:,2
H2O_cluster_allowed_cores:,2
H2O_cluster_status:,"locked, healthy"


In [6]:
# Load data
data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target.rename('target')

In [7]:
# Create H2OFrame
df = h2o.H2OFrame(pd.concat([X, y], axis=1))

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [8]:
# Split into train/test
train, test = df.split_frame(ratios=[0.8], seed=42)

### 3.1 Run H2O AutoML for Regression

In [9]:
aml_reg = H2OAutoML(
    max_runtime_secs=300,
    max_models=20,
    seed=42,
    nfolds=5,
    project_name="california_regression"
)
aml_reg.train(x=X.columns.tolist(), y='target', training_frame=train)

AutoML progress: |███████████████████████████████████████████████████████████████| (done) 100%


Model Details
=============
H2OGradientBoostingEstimator : Gradient Boosting Machine
Model Key: GBM_4_AutoML_1_20260728_172128


Model Summary: 
    number_of_trees    number_of_internal_trees    model_size_in_bytes    min_depth    max_depth    mean_depth    min_leaves    max_leaves    mean_leaves
--  -----------------  --------------------------  ---------------------  -----------  -----------  ------------  ------------  ------------  -------------
    99                 99                          277089                 10           10           10            41            451           218.404

ModelMetricsRegression: gbm
** Reported on train data. **

MSE: 0.07397325656727911
RMSE: 0.2719802503257895
MAE: 0.18538668936475916
RMSLE: 0.0835524573825608
Mean Residual Deviance: 0.07397325656727911

ModelMetricsRegression: gbm
** Reported on cross-validation data. **

MSE: 0.20714453019161605
RMSE: 0.4551313329047079
MAE: 0.29676202261609314
RMSLE: 0.13681969786908746
Mean Residual Deviance: 0.20714453019161605

Cross-Validation Metrics Summary: 
                        mean      sd           cv_1_valid    cv_2_valid    cv_3_valid    cv_4_valid    cv_5_valid
----------------------  --------  -----------  ------------  ------------  ------------  ------------  ------------
aic                     nan       0            nan           nan           nan           nan           nan
loglikelihood           nan       0            nan           nan           nan           nan           nan
mae                     0.296732  0.00322323   0.299136      0.292047      0.294765      0.298175      0.299539
mean_residual_deviance  0.207227  0.00613069   0.217706      0.204902      0.203079      0.207449      0.203
mse                     0.207227  0.00613069   0.217706      0.204902      0.203079      0.207449      0.203
r2                      0.844695  0.00317222   0.840185      0.845816      0.847472      0.842682      0.847321
residual_deviance       0.207227  0.00613069   0.217706      0.204902      0.203079      0.207449      0.203
rmse                    0.455183  0.00668183   0.46659       0.452661      0.450643      0.455466      0.450555
rmsle                   0.136859  0.000486415  0.137507      0.136554      0.136472      0.136504      0.137256

Scoring History: 
     timestamp            duration    number_of_trees    training_rmse        training_mae         training_deviance
---  -------------------  ----------  -----------------  -------------------  -------------------  -------------------
     2026-07-28 17:23:49  14.209 sec  0.0                1.1550860447424607   0.9126682923030268   1.334223770758782
     2026-07-28 17:23:49  14.321 sec  5.0                0.8099859863741191   0.6333660448978525   0.6560772981224546
     2026-07-28 17:23:49  14.430 sec  10.0               0.617830894441751    0.4722230808231802   0.38171501412669406
     2026-07-28 17:23:49  14.532 sec  15.0               0.5109753197147372   0.3784870497227261   0.26109577735757794
     2026-07-28 17:23:50  14.633 sec  20.0               0.44111368507007787  0.3156783019546691   0.19458128315610382
     2026-07-28 17:23:50  14.731 sec  25.0               0.4067439923962636   0.284500376285093    0.1654406753504517
     2026-07-28 17:23:50  14.828 sec  30.0               0.382227147884688    0.2624152441257563   0.14609759258006313
     2026-07-28 17:23:50  14.921 sec  35.0               0.3651895115355381   0.24789120218978403  0.1333633793355649
     2026-07-28 17:23:50  15.017 sec  40.0               0.34783585389151034  0.23473247741922482  0.12098978125243612
     2026-07-28 17:23:50  15.123 sec  45.0               0.3358994029907933   0.22632615955146831  0.11282840892957134
---  ---                  ---         ---                ---                  ---                  ---
     2026-07-28 17:23:50  15.353 sec  55.0               0.3176689889023286   0.21408124998465275  0.10091358651022775
     2026-07-28 17:23:50  15.452 sec  60.0              

In [10]:
# Show leaderboard
df_leader_reg = aml_reg.leaderboard.as_data_frame()
print(df_leader_reg.head())

                         model_id      rmse       mse       mae     rmsle  \
0  GBM_4_AutoML_1_20260728_172128  0.455131  0.207145  0.296762  0.136820   
1  GBM_2_AutoML_1_20260728_172128  0.455992  0.207929  0.300417  0.137703   
2  GBM_3_AutoML_1_20260728_172128  0.458074  0.209832  0.301015  0.137618   
3  GBM_1_AutoML_1_20260728_172128  0.459199  0.210863  0.302926  0.138295   
4  GBM_5_AutoML_1_20260728_172128  0.460024  0.211622  0.305412  0.139356   

   mean_residual_deviance  
0                0.207145  
1                0.207929  
2                0.209832  
3                0.210863  
4                0.211622  


/usr/local/lib/python3.12/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [11]:
# Evaluate on test
perf_reg = aml_reg.leader.model_performance(test)
print(f"H2O Regression R²: {perf_reg.r2():.4f}")
print(f"H2O Regression RMSE: {perf_reg.rmse():.4f}")

H2O Regression R²: 0.8523
H2O Regression RMSE: 0.4417


## 4. Classification Example: Breast Cancer Dataset

In [12]:
from sklearn.datasets import load_breast_cancer

In [13]:
# Load and prepare dataset
data_cls = load_breast_cancer(as_frame=True)
Xc = data_cls.data
yc = data_cls.target.rename('target')

In [14]:
df_cls = h2o.H2OFrame(pd.concat([Xc, yc], axis=1))
train_cls, test_cls = df_cls.split_frame(ratios=[0.7], seed=42)

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


### 4.1 Run H2O AutoML for Classification

In [15]:
aml_cls = H2OAutoML(
    max_runtime_secs=300,
    max_models=20,
    seed=42,
    nfolds=5,
    balance_classes=True,
    project_name="breast_cancer_classification"
)
aml_cls.train(x=Xc.columns.tolist(), y='target', training_frame=train_cls)

AutoML progress: |
17:28:39.574: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.


17:28:41.59: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.


17:28:41.407: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.

█
17:28:42.480: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.

███
17:28:43.764: _response param, We have det

key,value
Stacking strategy,cross_validation
Number of base models (used / total),4/5
# GBM base models (used / total),1/1
# XGBoost base models (used / total),1/1
# DRF base models (used / total),1/2
# GLM base models (used / total),1/1
Metalearner algorithm,GLM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5
Metalearner fold_column,None


In [16]:
# Show leaderboard
df_leader_cls = aml_cls.leaderboard.as_data_frame()
print(df_leader_cls.head())

                                            model_id      rmse       mse  \
0  StackedEnsemble_BestOfFamily_1_AutoML_2_202607...  0.178552  0.031881   
1  StackedEnsemble_AllModels_1_AutoML_2_20260728_...  0.183572  0.033699   
2       GBM_grid_1_AutoML_2_20260728_172839_model_22  0.185965  0.034583   
3       GBM_grid_1_AutoML_2_20260728_172839_model_11  0.186716  0.034863   
4       GBM_grid_1_AutoML_2_20260728_172839_model_20  0.188528  0.035543   

        mae     rmsle  mean_residual_deviance  
0  0.099521  0.128031                0.031881  
1  0.095885  0.131159                0.033699  
2  0.096905  0.131893                0.034583  
3  0.089201  0.134120                0.034863  
4  0.090711  0.133354                0.035543  


/usr/local/lib/python3.12/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [17]:
# Evaluate on test
perf_cls = aml_cls.leader.model_performance(test_cls)
print(f"H2O Classification: {perf_cls}")

H2O Classification: ModelMetricsRegressionGLM: stackedensemble
** Reported on test data. **

MSE: 0.03687836387225242
RMSE: 0.19203740227427682
MAE: 0.102186978783904
RMSLE: 0.12966219676564422
Mean Residual Deviance: 0.03687836387225242
R^2: 0.836282180057665
Null degrees of freedom: 177
Residual degrees of freedom: 173
Null deviance: 40.43221198186829
Residual deviance: 6.564348769260931
AIC: -70.28106575860501


MSE (Mean Squared Error):
- Measures the average squared difference between predicted and actual values. Lower values indicate better performance.


RMSE (Root Mean Squared Error):
- The square root of MSE, providing an error metric in the same units as the target variable. Lower values are better.


MAE (Mean Absolute Error):
-Measures the average absolute difference between predicted and actual values. Lower values indicate better performance.


RMSLE (Root Mean Squared Logarithmic Error):
- Similar to RMSE but uses the logarithm of the values, making it more robust to outliers. Lower values are better.


Mean Residual Deviance:
- Measures the goodness of fit of the model. Lower values indicate a better fit.


R² (Coefficient of Determination):
- Represents the proportion of variance explained by the model. Values closer to 1 indicate better performance.


Null Degrees of Freedom:
- The number of observations minus 1.


Residual Degrees of Freedom:
- The number of observations minus the number of parameters estimated.


Null Deviance:
- The deviance of the null model (model with no predictors).


Residual Deviance:
- The deviance of the fitted model. Lower values indicate a better fit.


AIC (Akaike Information Criterion):
- A measure of model quality, balancing goodness of fit and model complexity. Lower values indicate a better model.

## 5. Interpreting Results
- **Leaderboard** displays model ranking by default metric.
- Use `model_performance` to compute custom metrics (RMSE, R², AUC, accuracy, etc.).
- Top models are automatically ensembled by H2O's Stacked Ensemble.

## 6. AutoML Best Practices
- **Set time and model limits** (`max_runtime_secs`, `max_models`) to control cost and runtime.
- **Use cross-validation** (`nfolds`) for robust performance estimates.
- **Balance classes** for imbalanced classification tasks.
- **Review variable importance** on the leader model: `aml.leader.varimp()`.
- **Save and deploy** the best model: `h2o.save_model(aml.leader, path='best_model')`.

## 7. Exercise: Custom Dataset AutoML

**Task:** Apply H2O AutoML to a custom dataset.

1. Load any tabular dataset (CSV or from `sklearn.datasets`).
2. Decide whether it's a regression or classification task.
3. Initialize H2O and convert to `H2OFrame`.
4. Split into appropriate train/test ratios.
5. Run `H2OAutoML` with:
   - `max_runtime_secs=300`
   - `max_models=15`
   - `nfolds=5`
   - `balance_classes=True` (if classification)
6. Display the leaderboard and evaluate on the test set using relevant metrics.
7. Save the leaderboard to a pandas DataFrame and export it as `leaderboard.csv`

In [18]:
from sklearn.datasets import load_diabetes
import pandas as pd

# Load dataset
data = load_diabetes(as_frame=True)

# Features and target
X = data.data
y = data.target.rename("target")

# Combine into one DataFrame
df = pd.concat([X, y], axis=1)

df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [19]:
# Convert pandas DataFrame to H2OFrame
hf = h2o.H2OFrame(df)

# Split into train and test sets (80% train, 20% test)
train, test = hf.split_frame(ratios=[0.8], seed=42)

# Check rows
print("Train rows:", train.nrows)
print("Test rows :", test.nrows)

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Train rows: 343
Test rows : 99


**H2O Auto ML**

In [20]:
from h2o.automl import H2OAutoML

# Create AutoML object
aml = H2OAutoML(
    max_runtime_secs=300,
    max_models=15,
    seed=42,
    nfolds=5
)

# Train AutoML
aml.train(
    x=X.columns.tolist(),
    y="target",
    training_frame=train
)

AutoML progress: |███████████████████████████████████████████████████████████████| (done) 100%


key,value
Stacking strategy,cross_validation
Number of base models (used / total),2/6
# GBM base models (used / total),1/1
# XGBoost base models (used / total),0/1
# GLM base models (used / total),1/1
# DRF base models (used / total),0/2
# DeepLearning base models (used / total),0/1
Metalearner algorithm,GLM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5


# Leaderboard

In [21]:
# Show leaderboard
leaderboard = aml.leaderboard.as_data_frame()

print(leaderboard.head())

                                            model_id       rmse          mse  \
0  StackedEnsemble_BestOfFamily_1_AutoML_3_202607...  55.609540  3092.420886   
1                     GLM_1_AutoML_3_20260728_173821  55.630372  3094.738250   
2  StackedEnsemble_AllModels_1_AutoML_3_20260728_...  55.766459  3109.897951   
3                     GBM_3_AutoML_3_20260728_173821  56.521374  3194.665720   
4        GBM_grid_1_AutoML_3_20260728_173821_model_1  56.700131  3214.904893   

         mae     rmsle  mean_residual_deviance  
0  45.742663  0.427769             3092.420886  
1  45.588339  0.430390             3094.738250  
2  45.721363  0.425927             3109.897951  
3  46.123980  0.432413             3194.665720  
4  46.063954  0.428702             3214.904893  


/usr/local/lib/python3.12/dist-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


# Evaluation

In [22]:
# Evaluate best model
perf = aml.leader.model_performance(test)

print(f"R² : {perf.r2():.4f}")
print(f"RMSE : {perf.rmse():.4f}")
print(f"MAE : {perf.mae():.4f}")

R² : 0.5306
RMSE : 56.2820
MAE : 44.3384


In [23]:
# Save leaderboard
leaderboard.to_csv("leaderboard.csv", index=False)

print("Leaderboard saved successfully!")

Leaderboard saved successfully!
